# PdfExtractor — Google Colab GitHub Runner

Notebook ini hanya berfungsi sebagai **runner**. Source code pemrosesan tetap modular dan disimpan di repository GitHub; PDF dan semua hasil ekstraksi disimpan di Google Drive.

**Alur:** mount Drive → clone source GitHub → pilih runtime CPU/GPU → instal dependensi → validasi backend → ekstrak seluruh PDF → simpan database SQLite/CSV/JSONL dan aset.

> **Sebelum menjalankan:** unggah *isi* folder project `PdfExtractor` ke root repository GitHub Anda (pastikan `run_extraction.py`, `requirements.txt`, folder `PdfExtractor/`, dan `README.md` terlihat langsung di root). Repository public paling sederhana untuk `git clone` tanpa token.

Mode default adalah `gpu` untuk runtime NVIDIA Colab. Pada Colab, pilih **Runtime → Change runtime type → T4 GPU** (atau GPU NVIDIA lain yang tersedia) sebelum menjalankan notebook. Bila ingin CPU, ubah `DEVICE_MODE` di sel konfigurasi menjadi `cpu`.

## 1. Konfigurasi — edit sel ini terlebih dahulu

Ganti URL GitHub, nama folder PDF, dan lokasi output sesuai Drive Anda. Jangan menaruh PDF atau hasil ekstraksi di repository GitHub.

In [ ]:
# ===== GANTI NILAI DI BAGIAN INI =====
GITHUB_REPO_URL = "https://github.com/zuudevs/PdfExtractor.git"
GITHUB_BRANCH = "main"
PROJECT_SUBDIR = ""  # Kosong jika run_extraction.py berada di root repo; selain itu isi subfolder, mis. "src"

# Runtime: "gpu" untuk NVIDIA CUDA di Colab; "cpu" untuk mode CPU.
DEVICE_MODE = "gpu"

# Tier: "flash", "basic", "standard", atau "advanced".
# Standard menggunakan VLM dan butuh memori/engine yang kompatibel.
TIER = "standard"
OCR_MODE = "auto"  # "auto", "txt", atau "ocr"
MODEL_SOURCE = "auto"  # "auto", "huggingface", atau "modelscope"

INPUT_DIR = "/content/drive/MyDrive/papers"
OUTPUT_DIR = "/content/drive/MyDrive/output/extracted_pdf"

# True pada run awal atau bila model belum tersedia di runtime saat ini.
# Cache model default Colab tidak persisten ketika runtime dibuang.
PREPARE_MODELS = True
RESUME_VALID_RESULTS = True
STOP_ON_ERROR = False

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Periksa pilihan GPU/CPU sebelum instalasi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if DEVICE_MODE not in {"cpu", "gpu"}:
    raise ValueError("DEVICE_MODE harus 'cpu' atau 'gpu'.")
if TIER not in {"flash", "basic", "standard", "advanced"}:
    raise ValueError("TIER tidak valid.")
if MODEL_SOURCE not in {"auto", "huggingface", "modelscope"}:
    raise ValueError("MODEL_SOURCE tidak valid.")

if DEVICE_MODE == "gpu":
    # Cek driver/runtime tanpa mengimpor PyTorch sebelum paket dipasang.
    result = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "GPU NVIDIA tidak terdeteksi. Di Colab, buka Runtime > Change runtime type > pilih GPU, lalu restart runtime. "
            "Jika ingin CPU, ubah DEVICE_MODE='cpu' di sel konfigurasi."
        )
    print(result.stdout[:5000])
else:
    print("Mode CPU dipilih. Ekstraksi tetap dapat memerlukan RAM yang besar untuk tier standard/advanced.")

input_path = Path(INPUT_DIR).expanduser()
output_path = Path(OUTPUT_DIR).expanduser()
if not input_path.exists() or not input_path.is_dir():
    raise FileNotFoundError(f"Folder PDF tidak ditemukan: {input_path}. Perbaiki INPUT_DIR.")
input_resolved = input_path.resolve()
output_resolved = output_path.resolve()
if input_resolved == output_resolved or input_resolved in output_resolved.parents:
    raise ValueError("OUTPUT_DIR harus terpisah dan tidak boleh berada di dalam INPUT_DIR.")

pdf_files = sorted(input_path.rglob("*.pdf"))
# PDF extensions sometimes use uppercase .PDF.
pdf_files += sorted(p for p in input_path.rglob("*.PDF") if p not in pdf_files)
if not pdf_files:
    raise FileNotFoundError(f"Tidak ditemukan file PDF di {input_path} atau subfoldernya.")
print(f"Jumlah PDF ditemukan: {len(pdf_files)}")
print(f"Mode: {DEVICE_MODE.upper()} | Tier: {TIER} | OCR: {OCR_MODE}")
print(f"Input : {input_path}")
print(f"Output: {output_path}")

## 4. Clone atau perbarui source dari GitHub

In [ ]:
import shutil

if not GITHUB_REPO_URL.startswith("https://github.com/") or "YOUR_USERNAME" in GITHUB_REPO_URL:
    raise ValueError(
        "Isi GITHUB_REPO_URL dengan URL repository Anda, mis. https://github.com/nama/pdf-extractor.git"
    )

REPO_DIR = Path("/content/PdfExtractor")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO_DIR)]
print("Menjalankan:", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

PROJECT_DIR = (REPO_DIR / PROJECT_SUBDIR).resolve() if PROJECT_SUBDIR else REPO_DIR.resolve()
if not PROJECT_DIR.is_relative_to(REPO_DIR.resolve()):
    raise ValueError("PROJECT_SUBDIR harus berada di dalam repository hasil clone.")

required_paths = [PROJECT_DIR / "run_extraction.py", PROJECT_DIR / "requirements.txt", PROJECT_DIR / "PdfExtractor"]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Struktur repository tidak sesuai. File/folder ini tidak ditemukan:\n- " + "\n- ".join(missing) +
        "\nUnggah isi project ke root repo, atau atur PROJECT_SUBDIR dengan benar."
    )
print("Source siap:", PROJECT_DIR)
print("File entry point:", PROJECT_DIR / "run_extraction.py")

## 5. Instal dependensi sesuai runtime

In [ ]:
# Base requirements memasang MinerU 4.x dan pypdf.
# Pada mode GPU, tambahkan extra resmi [full] untuk Torch + engine GPU yang didukung platform.
install_cmd = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")]
if DEVICE_MODE == "gpu":
    install_cmd.append("mineru[full]>=4.0,<5")

print("Perintah instalasi:", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)
print("Instalasi selesai. Bila Colab menampilkan notifikasi untuk restart runtime, restart lalu jalankan ulang notebook dari awal.")

## 6. Validasi backend aktual sebelum memproses PDF

In [ ]:
import importlib.util
import json

# Set konfigurasi yang sama pada proses validasi. Source modular mengulang konfigurasi ini
# di dalam run_extraction.py sebelum MinerU diimpor.
validation_env = os.environ.copy()
if DEVICE_MODE == "cpu":
    validation_env["MINERU_MODEL_SMALL_BACKEND"] = "onnx"
    validation_env["MINERU_MODEL_VLM_ENGINE"] = "llama-cpp"
    validation_env["MINERU_TABLE_DEVICE"] = "cpu"
    validation_env["CUDA_VISIBLE_DEVICES"] = "-1"
else:
    validation_env.pop("CUDA_VISIBLE_DEVICES", None) if validation_env.get("CUDA_VISIBLE_DEVICES") == "-1" else None
    validation_env["MINERU_MODEL_SMALL_BACKEND"] = "torch"
    validation_env["MINERU_TABLE_DEVICE"] = "cuda"
    if TIER in {"standard", "advanced"}:
        validation_env["MINERU_MODEL_VLM_ENGINE"] = "vllm"
    else:
        validation_env.pop("MINERU_MODEL_VLM_ENGINE", None)

if DEVICE_MODE == "gpu":
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch tidak mendeteksi CUDA. Gunakan runtime GPU NVIDIA yang kompatibel atau ubah DEVICE_MODE='cpu'.")
    if TIER in {"standard", "advanced"} and importlib.util.find_spec("vllm") is None:
        raise RuntimeError("vLLM tidak terpasang setelah instalasi MinerU full. Periksa dukungan wheel/runtime Colab.")

print("Konfigurasi yang diperiksa:")
for key in ["MINERU_MODEL_SMALL_BACKEND", "MINERU_MODEL_VLM_ENGINE", "MINERU_TABLE_DEVICE", "CUDA_VISIBLE_DEVICES"]:
    print(f"  {key} = {validation_env.get(key, '(unset)')}")

show_result = subprocess.run(
    ["mineru-kit", "models", "show"],
    cwd=str(PROJECT_DIR), env=validation_env, text=True, capture_output=True
)
print(show_result.stdout)
if show_result.stderr:
    print(show_result.stderr[-4000:])
if show_result.returncode != 0:
    raise RuntimeError("mineru-kit models show gagal. Periksa log di atas sebelum melanjutkan.")
print("Pemeriksaan backend selesai. Status model juga akan diverifikasi jika PREPARE_MODELS=True.")

## 7. Jalankan ekstraksi batch

In [ ]:
run_cmd = [
    sys.executable,
    str(PROJECT_DIR / "run_extraction.py"),
    "--input-dir", str(input_path),
    "--output-dir", str(output_path),
    "--device", DEVICE_MODE,
    "--tier", TIER,
    "--ocr-mode", OCR_MODE,
    "--model-source", MODEL_SOURCE,
]
if PREPARE_MODELS:
    run_cmd.append("--prepare-models")
if not RESUME_VALID_RESULTS:
    run_cmd.append("--no-resume")
if STOP_ON_ERROR:
    run_cmd.append("--stop-on-error")

print("Perintah:", " ".join(run_cmd), flush=True)
print("Mulai pemrosesan. Mode standard/advanced dapat mengunduh model besar pada penggunaan pertama.", flush=True)
run_result = subprocess.run(run_cmd, cwd=str(PROJECT_DIR), env=os.environ.copy())
if run_result.returncode != 0:
    raise RuntimeError(
        f"Pipeline keluar dengan exit code {run_result.returncode}. Periksa log error di atas dan folder documents/*/_ERROR.json."
    )
print("Proses ekstraksi selesai.")

## 8. Periksa hasil dan audit

In [ ]:
import csv

DB_DIR = output_path / "database"
expected_outputs = [
    DB_DIR / "literature_database.sqlite",
    DB_DIR / "literature_documents.csv",
    DB_DIR / "literature_tables.csv",
    DB_DIR / "literature_figures.csv",
    DB_DIR / "literature_text_blocks.csv",
    DB_DIR / "literature_elements.jsonl",
    DB_DIR / "literature_extraction_audit.csv",
    DB_DIR / "literature_image_inventory.csv",
]
print("Lokasi output:", output_path)
for item in expected_outputs:
    print(("[OK] " if item.exists() else "[BELUM ADA] ") + str(item))

audit_file = DB_DIR / "literature_extraction_audit.csv"
if audit_file.exists():
    with audit_file.open("r", encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    print(f"\\nBaris audit: {len(rows)}")
    if rows:
        print("Kolom audit:", list(rows[0].keys()))
        for row in rows[:10]:
            print({k: row.get(k) for k in list(row)[:6]})

print("\\nDatabase dan seluruh aset disimpan di Google Drive, bukan pada repository GitHub.")

## Catatan operasional

- Notebook mengharapkan repository GitHub memiliki `run_extraction.py`, `requirements.txt`, dan folder `PdfExtractor/` di root (atau atur `PROJECT_SUBDIR`).
- `DEVICE_MODE="gpu"` ditujukan untuk GPU NVIDIA CUDA di Linux/Colab, bukan Radeon AMD pada Windows. Pada GPU T4, runtime dan versi paket tetap harus kompatibel; notebook melakukan pemeriksaan GPU, vLLM, dan pemilihan backend sebelum memulai ekstraksi.
- `standard` membutuhkan VLM; instalasi engine GPU dapat gagal bila image Colab atau wheel tidak kompatibel. Bila instalasi GPU tidak berhasil, jangan lanjut ke ekstraksi. Periksa output instalasi, atau ubah ke `DEVICE_MODE="cpu"` untuk paket dasar.
- `PREPARE_MODELS=True` mengunduh dan memverifikasi model yang cocok pada penggunaan pertama. Cache model Colab umumnya hilang ketika runtime dibuang, sehingga pengunduhan bisa terulang pada sesi baru.
- Database, Markdown, JSON, CSV/JSONL, audit, dan aset gambar ditulis ke `OUTPUT_DIR` di Drive. Periksa `literature_extraction_audit.csv` dan bandingkan sampel hasil dengan PDF asli sebelum menggunakan hasil untuk analisis ilmiah.
- Dokumentasi resmi: [MinerU Tiers and Runtimes](https://opendatalab.github.io/MinerU/usage/tiers/), [Extension Modules](https://opendatalab.github.io/MinerU/quick_start/extension_modules/), [Model Source](https://opendatalab.github.io/MinerU/usage/model_source/).